## Imports

In [ ]:
import os
import sys
import warnings
from pyspark.sql import SparkSession

warnings.filterwarnings("ignore", category=UserWarning)

os.environ["SPARK_CONNECT_MODE_ENABLED"] = "0"

if sys.platform == "darwin":
    os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
elif sys.platform.startswith("linux"):
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Bigdata") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("Spark session started ✅")

In [ ]:
base_path = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
base_path = os.path.join(base_path, "data")

parquets = ["orders", "customers", "order_items", "payments", "products", "sellers", "reviews"]

dataframes = {}
i = 0
for parquet in parquets:
    df = spark.read.parquet(f"{os.path.join(base_path, "2_silver", parquet)}")
    dataframes[parquet] = df
    i += 1
print(f"{i} dataframes successfully loaded ✅")

## Gold tables

### Sales KPIs (total and monthly CA, average cart, payments distribution)

In [ ]:
from pyspark.sql import functions as F

df_sales = dataframes["orders"].filter(F.col("order_status") == "delivered") \
    .join(dataframes["payments"], on="order_id", how="inner") \
    .join(dataframes["order_items"], on="order_id", how="inner") \
    .withColumn("month_year", F.date_format("order_purchase_timestamp", "yyyy-MM")) \
    .select(
        "order_id", 
        "customer_id", 
        "payment_type", 
        "payment_value",
        "price",
        "month_year"
    )

### Delivery and reviews KPIs (delivery delay, lateness rate, average rate)

In [ ]:
df_deliveries_reviews = dataframes["orders"].filter(F.col("order_status") == "delivered") \
    .withColumn("delay_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .withColumn("is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), True).otherwise(False)) \
    .select("order_id", "delay_days", "is_late") \
    .join(dataframes["reviews"], on="order_id", how="inner") \
    .select("order_id", "delay_days", "is_late", "review_score")


### Tops KPIs (top sellers, top categories)

In [ ]:
df_products_sellers = dataframes["order_items"] \
    .join(dataframes["products"], on="product_id", how="inner") \
    .join(dataframes["sellers"], on="seller_id", how="inner") \
    .select("order_id", "product_id", "product_category_name_english", "price", "seller_id", "seller_city", "seller_state")

### Geo customers KPIs (to be joined to df_sales to get CA by city or state)

In [ ]:
df_geo_customers = dataframes["orders"] \
    .join(dataframes["customers"], on="customer_id", how="inner") \
    .select("order_id", "customer_id", "customer_zip_code_prefix", "customer_city", "customer_state")

## KPIs

In [ ]:
df_items_unique = df_sales.select("order_id", "month_year", "price").distinct()

# Total CA
total_ca = df_items_unique.agg(F.sum("price").alias("total_CA"))

print("=== TOTAL CA ===")
total_ca.show()

# Monthly CA
monthly_ca = df_items_unique.groupBy("month_year") \
    .agg(F.sum("price").alias("monthly_CA")) \
    .orderBy("month_year")

print("=== MONTHLY CA ===")
monthly_ca.show()

# Average cart
avg_cart = df_items_unique.groupBy("order_id") \
    .agg(F.sum("price").alias("order_total")) \
    .agg(F.avg("order_total").alias("average_cart"))

print("=== AVERAGE CART ===")
avg_cart.show()

# Payment distribution
payment_dist = df_sales.select("order_id", "payment_type").distinct() \
    .groupBy("payment_type") \
    .count() \
    .orderBy("count", ascending=False)

print("=== PAYMENT DISTRIBUTION ===")
payment_dist.show()

In [ ]:
# Annual CA
annual_ca = df_sales.withColumn("year", F.year("month_year")) \
    .groupBy("year") \
    .agg(F.sum("payment_value").alias("annual_CA")) \
    .orderBy("year")
print("=== ANNUAL CA ===")
annual_ca.show()

In [ ]:
# Average delivery delay
avg_delay = df_deliveries_reviews.agg(
    F.avg("delay_days").alias("avg_delivery_days")
)
print("=== AVERAGE DELIVERY DELAY ===")
avg_delay.show()

# Lateness rate
lateness_rate = df_deliveries_reviews.agg(
    (F.sum(F.col("is_late").cast("integer")) / F.count("*") * 100).alias("lateness_rate_%")
)
print("=== LATENESS RATE ===")
lateness_rate.show()

# Average review score
avg_score = df_deliveries_reviews.agg(
    F.avg("review_score").alias("avg_review_score")
)
print("=== AVERAGE REVIEW SCORE ===")
avg_score.show()

# Impact of lateness on satisfaction
impact = df_deliveries_reviews.groupBy("is_late") \
    .agg(F.avg("review_score").alias("avg_score")) \
    .orderBy("is_late")
print("=== IMPACT OF LATENESS ON SATISFACTION ===")
impact.show()

In [ ]:
# Annual CA with percentage
total = df_sales.agg(F.sum("payment_value")).collect()[0][0]

annual_ca_pct = df_sales.withColumn("year", F.year("month_year")) \
    .groupBy("year") \
    .agg(F.sum("payment_value").alias("annual_CA")) \
    .withColumn("percentage", F.round(F.col("annual_CA") / total * 100, 2)) \
    .orderBy("year")

print("=== ANNUAL CA WITH PERCENTAGE ===")
annual_ca_pct.show()

In [ ]:
# Payment distribution with percentage
total_payments = df_sales.count()

payment_dist_pct = df_sales.groupBy("payment_type") \
    .count() \
    .withColumn("percentage", F.round(F.col("count") / total_payments * 100, 2)) \
    .orderBy("count", ascending=False)

print("=== PAYMENT DISTRIBUTION WITH PERCENTAGE ===")
payment_dist_pct.show()

# Monthly CA with percentage
monthly_ca_pct = df_sales.groupBy("month_year") \
    .agg(F.sum("payment_value").alias("monthly_CA")) \
    .withColumn("percentage", F.round(F.col("monthly_CA") / total * 100, 2)) \
    .orderBy("month_year")

print("=== MONTHLY CA WITH PERCENTAGE ===")
monthly_ca_pct.show()

# Impact of lateness with percentage
total_orders = df_deliveries_reviews.count()
impact_pct = df_deliveries_reviews.groupBy("is_late") \
    .agg(
        F.count("*").alias("count"),
        F.avg("review_score").alias("avg_score")
    ) \
    .withColumn("percentage", F.round(F.col("count") / total_orders * 100, 2)) \
    .orderBy("is_late")

print("=== IMPACT OF LATENESS WITH PERCENTAGE ===")
impact_pct.show()

In [ ]:
# Top 10 categories by CA
top_categories = df_products_sellers \
    .groupBy("product_category_name_english") \
    .agg(F.sum("price").alias("CA")) \
    .orderBy("CA", ascending=False)

print("=== TOP 10 CATEGORIES ===")
top_categories.show(10)

# Top 10 sellers by CA
top_sellers = df_products_sellers \
    .groupBy("seller_id", "seller_city", "seller_state") \
    .agg(F.sum("price").alias("CA")) \
    .orderBy("CA", ascending=False)

print("=== TOP 10 SELLERS ===")
top_sellers.show(10)

In [ ]:
# Total CA for percentage calculation
total_product_CA = df_products_sellers.agg(F.sum("price")).collect()[0][0]

# Top categories with percentage
top_categories = df_products_sellers \
    .groupBy("product_category_name_english") \
    .agg(F.sum("price").alias("CA")) \
    .withColumn("percentage", F.round(F.col("CA") / total_product_CA * 100, 2)) \
    .orderBy("CA", ascending=False)

# Top sellers with percentage
top_sellers = df_products_sellers \
    .groupBy("seller_id", "seller_city", "seller_state") \
    .agg(F.sum("price").alias("CA")) \
    .withColumn("percentage", F.round(F.col("CA") / total_product_CA * 100, 2)) \
    .orderBy("CA", ascending=False)

print("=== TOP 10 CATEGORIES WITH PERCENTAGE ===")
top_categories.show(10)

print("=== TOP 10 SELLERS WITH PERCENTAGE ===")
top_sellers.show(10)

In [ ]:
avg_cart = df_sales.groupBy("order_id") \
    .agg(F.sum("payment_value").alias("order_total")) \
    .agg(F.avg("order_total").alias("average_cart"))

print("=== AVERAGE CART ===")
avg_cart.show()

## Exports

In [ ]:
gold_dataframes = {
    "sales": df_sales,
    "deliveries_reviews": df_deliveries_reviews,
    "products_sellers": df_products_sellers,
    "geo_customers": df_geo_customers
}

for name, df in gold_dataframes.items():
    df.write.mode("overwrite").parquet(f"{base_path}/3_gold/{name}")
    print(f"Dataframe {name} saved to gold ✅")